# Dynamic Binding, Quantificational Subordination and _different_

This lambda notebook contains a demonstration of some ideas found in the following papers:

- Gotham, Matthew (2019). [Quantificational Subordination as Anaphora to a Function](https://matthewgotham.github.io/linguistics/#FG2019). In Raffaella Bernardi, Greg Kobele & Sylvain Pogodalla (eds.), _Formal Grammar: FG 2019_, 51–66. Lecture Notes in Computer Science 11668. Berlin, Heidelberg: Springer.
- Gotham, Matthew (2018). [A Model-Theoretic Reconstruction of Type-Theoretic Semantics for Anaphora](https://matthewgotham.github.io/linguistics/#FG2017). In Annie Foret, Reinhard Muskens & Sylvain Pogodalla (eds.), _Formal Grammar: FG 2017_, 37–53. Lecture Notes in Computer Science 10686. Berlin, Heidelberg: Springer.

The final section, on _different_, contains ideas that have not been published yet.

In [1]:
composition_system = lang.hk_system.copy()
meta.get_type_system().add_atomic(types.BasicType("v")) # events
display(meta.get_type_system())
lang.get_system()

Type system with atomic types: $n, t, v, e$

Composition system 'Type-driven composition'
Operations: {FA, PM, PA, VAC}

## Preliminary Remarks on the Implementation

### Type Variables

There is a some redudancy in the lexicon below because of a shortcoming in the type inference system for flexible types. Although the [documentation](https://github.com/rawlins/lambda-notebook/blob/master/notebooks/documentation/Intro%20to%20type%20variables.ipynb) suggests that each type variable should be &lsquo;bound&rsquo; within each lexical entry, that description does not match the behviour of the system in cases like the following.

In [2]:
%%lamb
||I|| = L x_X: x
||T|| = L x_X: L f_<X,Y>: f(x)

$[\![\text{\textbf{I}}]\!]^{}_{\left\langle{}X,X\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: {x}$<br />
$[\![\text{\textbf{T}}]\!]^{}_{\left\langle{}X,\left\langle{}\left\langle{}X,Y\right\rangle{},Y\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: \lambda{} f_{\left\langle{}X,Y\right\rangle{}} \: . \: {f}({x})$

In [3]:
(T * I)

Composition of "[T I]" failed:<br />
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:red">**TypeMismatch**</span>: type $\left\langle{}X,X\right\rangle{}$ and type $X$ conflict (Occurs check failure while trying to infer function type given argument type `<X,X>`)<br />
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:red">**TypeMismatch**</span>: type $\left\langle{}X,\left\langle{}\left\langle{}X,Y\right\rangle{},Y\right\rangle{}\right\rangle{}$ and type $X$ conflict (Occurs check failure while trying to infer function type given argument type `<X,<<X,Y>,Y>>`)<br />
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:red">**TypeMismatch**</span>: $[\![\text{\textbf{T}}]\!]^{}_{\left\langle{}X,\left\langle{}\left\langle{}X,Y\right\rangle{},Y\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: \lambda{} f_{\left\langle{}X,Y\right\rangle{}} \: . \: {f}({x})$ and $[\![\text{\textbf{I}}]\!]^{}_{\left\langle{}X,X\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: {x}$ conflict (Predicate Modification needs property types)<br />
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:red"><b>Composition failure</b></span> on: $[\![\text{\textbf{T}}]\!]^{}_{\left\langle{}X,\left\langle{}\left\langle{}X,Y\right\rangle{},Y\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: \lambda{} f_{\left\langle{}X,Y\right\rangle{}} \: . \: {f}({x})$ * $[\![\text{\textbf{I}}]\!]^{}_{\left\langle{}X,X\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: {x}$  (Predicate Abstraction requires a valid binder)<br />
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:red"><b>Composition failure</b></span> on: $[\![\text{\textbf{I}}]\!]^{}_{\left\langle{}X,X\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: {x}$ * $[\![\text{\textbf{T}}]\!]^{}_{\left\langle{}X,\left\langle{}\left\langle{}X,Y\right\rangle{},Y\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: \lambda{} f_{\left\langle{}X,Y\right\rangle{}} \: . \: {f}({x})$  (Predicate Abstraction requires a valid binder)<br />
&nbsp;&nbsp;&nbsp;&nbsp;<span style="color:red">**TypeMismatch**</span>: $[\![\text{\textbf{T}}]\!]^{}_{\left\langle{}X,\left\langle{}\left\langle{}X,Y\right\rangle{},Y\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: \lambda{} f_{\left\langle{}X,Y\right\rangle{}} \: . \: {f}({x})$ and $[\![\text{\textbf{I}}]\!]^{}_{\left\langle{}X,X\right\rangle{}} \:=\: \lambda{} x_{X} \: . \: {x}$ conflict (Vacuous composition needs at least one fully vacuous element)<br />


The expected behaviour is that you get two results:

1. $\lambda x_X.\lambda f_{\langle X,Y\rangle}.f(x)\qquad$  ($[\![\textbf{I}]\!]$ applied to $[\![\textbf{T}]\!]$), and
2. $\lambda f_{\langle\langle X,X\rangle,Y\rangle}.f(\lambda x_X.x)\quad$($[\![\textbf{T}]\!]$ applied to $[\![\textbf{I}]\!]$).

But this doesn't happen, presumably because the system is trying to unify the $X$ from $[\![\textbf{I}]\!]$ with the $X$ from $[\![\textbf{T}]\!]$. That means that in the following lexicon, I am forced to repeat entries in some places in order to keep variable names distinct across entries.

### (Un)Modified NPs

In the papers mentioned above, in order for both unmodified and modified NPs to have the same type pattern, there are lexical entries for nouns such as $[\![\textit{farmer}]\!] = \lambda i_X.\lambda u_{(e,1)}.Farmer(u[0])$ &mdash; i.e., with a dummy modification of the unit type. In the lexicon below, for the sake of simplicity of representation I have not done this; instead I have given (at least) two forms of lexical entry for items that nouns as an argument: one for the unmodified form (type pattern $\langle X,\langle e,t\rangle\rangle$), and one for the modified form (type pattern $\langle X,\langle(e,Y),t\rangle\rangle$).

### Determiners

For simplicity's sake, the entries for determiners given below are for weak readings only. Also, for now the only determiners we're interested in are monotone-increasing, so the lexical entries don't include the maximal participant condition clause that you would need for determiners that do not have this property. See the 2019 paper for the more complete version, and feel free to implement it yourself!

### Pronouns

In both papers, the list of possible pronouns is given an inductive definition, in terms of &lsquo;natural resolution functions&rsquo; (NRFs). In the following lexicon I have just enumerated pronoun meanings to account for the meanings that we have, but they should all **be** NRFs according to those definitions.

## Lexicon

Aside from the amendments noted above, this is the lexicon from the 2019 paper.

In [4]:
%%lamb reset, ambiguity
# nouns
||farmer|| = L i_X: Farmer_<e,t>
||farmers|| = L i_X: Farmer_<e,t>
||donkey|| = L i_X1: Donkey_<e,t>
||donkeys|| = L i_X1: Donkey_<e,t>
# verbs
||brays|| = L x_e: L i_X2: L e_v: Bray_<(e,v),t>(x,e)
||bray|| = L x_e: L i_X2: L e_v: Bray_<(e,v),t>(x,e)
||owns|| = L y_e: L x_e: L i_X2: L e_v: Own_<(e,e,v),t>(x,y,e)
||own|| = L y_e: L x_e: L i_X2: L e_v: Own_<(e,e,v),t>(x,y,e)
||feeds|| = L y_e: L x_e: L i_X2: L e_v: Feed_<(e,e,v),t>(x,y,e)
||feed|| = L y_e: L x_e: L i_X2: L e_v: Feed_<(e,e,v),t>(x,y,e)
# indefinites
||a|| = L n_<X3,<(e,Y),t>>: L v_<e,<(X3,(e,Y)),<Y2,t>>>: L i_X3: L o_((e,Y),Y2): n(i)(o[0]) & v(o[0][0])((i,o[0]))(o[1]) # modified N
||a|| = L n_<X3,<e,t>>: L v_<e,<(X3,e),<Y2,t>>>: L i_X3: L o_(e,Y2): n(i)(o[0]) & v(o[0])((i,o[0]))(o[1]) # unmodified N
||a|| = L n_<X4,<e,t>>: L v_<e,<(X4,e),<Y2,t>>>: L i_X4: L o_(e,Y2): n(i)(o[0]) & v(o[0])((i,o[0]))(o[1]) # unmodified N; solely because of bug
# relativizer
||who|| = L v_<e,<(X4,(e,Y3)),<Y4,t>>>: L n_<X4,<(e,Y3),t>>: L i_X4: L o_((e,Y3),Y4): n(i)(o[0]) & v(o[0][0])((i,o[0]))(o[1])
||who|| = L v_<e,<(X4,e),<Y4,t>>>: L n_<X4,<e,t>>: L i_X4: L o_(e,Y4): n(i)(o[0]) & v(o[0])((i,o[0]))(o[1])
# existential closure
||E|| = L p_<{e},<Y4,t>>: Exists m_Y4: p({})(m)
||C|| = L p_t: p # to eliminate undesired readings inherent in using so many polymorphic types
# determiners
det0 = L d_<({e},{e}),t>: L n_<X3,<(e,Y),t>>: L v_<e,<(X3,(e,Y)),<Y2,t>>>: L i_X3: L f_<(e,Y),Y2>: Dom(f) <= (Set o_(e,Y): n(i)(o)) & d((Set x_e: Exists y_Y: n(i)((x,y))),(Set x_e: Exists y_Y: (x,y) << Dom(f))) & (Forall o_(e,Y): (o << Dom(f)) ==> v(o[0])((i,o))(f(o)))
det1 = L d_<({e},{e}),t>: L n_<X3,<e,t>>: L v_<e,<(X3,e),<Y2,t>>>: L i_X3: L f_<e,Y2>: Dom(f) <= (Set z_e: n(i)(z)) & d((Set x_e: n(i)(x)),Dom(f)) & (Forall z_e: (z << Dom(f)) ==> v(z)((i,z))(f(z)))
# 
||every|| = det0(L a_({e},{e}): a[0] <= a[1])
||every|| = det1(L a_({e},{e}): a[0] <= a[1])
# 
||most|| = det0(L a_({e},{e}): Card_<{e},n>(a[0] & a[1]) * 2 > Card_<{e},n>(a[0]))
||most|| = det1(L a_({e},{e}): Card_<{e},n>(a[0] & a[1]) * 2 > Card_<{e},n>(a[0]))
#
||two|| = det0(L a_({e},{e}): Card_<{e},n>(a[0] & a[1]) == 2)
||two|| = det1(L a_({e},{e}): Card_<{e},n>(a[0] & a[1]) == 2)
# pronouns
||it|| = L v_<e,<(e,X5),<Z7,t>>>: L i_(e,X5): v(i[0])(i) # pron0
||it|| = L v_<e,<(X5,e),<Z7,t>>>: L i_(X5,e): v(i[1])(i) # pron1
||it|| = L v_<e,<(X5,(e,Y5)),<Z7,t>>>: L i_(X5,(e,Y5)): v(i[1][0])(i) # pron10
||it|| = L v_<e,<(X5,(Y5,e)),<Z7,t>>>: L i_(X5,(Y5,e)): v(i[1][1])(i) # pron11
||it|| = L v_<e,<(X5,(Y5,(e,Z8))),<Z7,t>>>: L i_(X5,(Y5,(e,Z1))): v(i[1][1][0])(i) # pron110
||it|| = L v_<e,<(X5,(Y5,(Z8,e))),<Z7,t>>>: L i_(X5,(Y5,(Z1,e))): v(i[1][1][1])(i) # pron111
# repeated
||he|| = L v_<e,<(e,X6),<Z8,t>>>: L i_(e,X6): v(i[0])(i) # pron0
||he|| = L v_<e,<(X6,e),<Z8,t>>>: L i_(X6,e): v(i[1])(i) # pron1
||he|| = L v_<e,<(X6,(e,Y6)),<Z8,t>>>: L i_(X6,(e,Y6)): v(i[1][0])(i) # pron10
||he|| = L v_<e,<(X6,(Y6,e)),<Z8,t>>>: L i_(X6,(Y6,e)): v(i[1][1])(i) # pron11
||he|| = L v_<e,<(X6,(Y6,(e,Z9))),<Z8,t>>>: L i_(X6,(Y6,(e,Z9))): v(i[1][1][0])(i) # pron110
||he|| = L v_<e,<(X6,(Y6,(Z9,e))),<Z8,t>>>: L i_(X6,(Y6,(Z9,e))): v(i[1][1][1])(i) # pron111
# connectives
||aND|| = L q_<(X9,Y9),<Z9,t>>: L p_<X9,<Y9,t>>: L i_X9: L o_(Y9,Z9): p(i)(o[0]) & q((i,o[0]))(o[1])
||iF|| = L p_<X,<Y,t>>: L q_<(X,Y),<Z,t>>: L i_X: L f_<Y,Z>: Dom(f) == (Set a_Y: p(i)(a)) & (Forall a_Y: (a << Dom(f)) ==> q((i,a))(f(a)))
# pronouns for quantificational subordination
||of_them|| = L i_<e,X7>: L x_e: x << Dom(i)
||of_them|| = L i_(X9,<e,X7>): L x_e: x << Dom(i[1])
||it|| = L v_<e,<(<e,(e,Y7)>,e),<Z6,t>>>: L i_(<e,(e,Y7)>,e): v((i[0](i[1]))[0])(i)
||it|| = L v_<e,<((X9,<e,(e,Y7)>),e),<Z6,t>>>: L i_((X9,<e,(e,Y7)>),e): v((i[0][1](i[1]))[0])(i)
# for telescoping
||it|| = L v_<e,<(X,(e,<e,(e,Z)>)),<Y,t>>>: L i_(X,(e,<e,(e,Z)>)): v((i[1][1](i[1][0]))[0])(i)

$[\![\text{\textbf{farmer}}]\!]^{}_{\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{X} \: . \: {Farmer}$<br />
$[\![\text{\textbf{farmers}}]\!]^{}_{\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{X} \: . \: {Farmer}$<br />
$[\![\text{\textbf{donkey}}]\!]^{}_{\left\langle{}X',\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{X'} \: . \: {Donkey}$<br />
$[\![\text{\textbf{donkeys}}]\!]^{}_{\left\langle{}X',\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{X'} \: . \: {Donkey}$<br />
$[\![\text{\textbf{brays}}]\!]^{}_{\left\langle{}e,\left\langle{}X'',\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{e} \: . \: \lambda{} i_{X''} \: . \: \lambda{} e_{v} \: . \: {Bray}({x}, {e})$<br />
$[\![\text{\textbf{bray}}]\!]^{}_{\left\langle{}e,\left\langle{}X'',\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} x_{e} \: . \: \lambda{} i_{X''} \: . \: \lambda{} e_{v} \: . \: {Bray}({x}, {e})$<br />
$[\![\text{\textbf{owns}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}X'',\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{X''} \: . \: \lambda{} e_{v} \: . \: {Own}({x}, {y}, {e})$<br />
$[\![\text{\textbf{own}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}X'',\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{X''} \: . \: \lambda{} e_{v} \: . \: {Own}({x}, {y}, {e})$<br />
$[\![\text{\textbf{feeds}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}X'',\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{X''} \: . \: \lambda{} e_{v} \: . \: {Feed}({x}, {y}, {e})$<br />
$[\![\text{\textbf{feed}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}X'',\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{X''} \: . \: \lambda{} e_{v} \: . \: {Feed}({x}, {y}, {e})$<br />
$[\![\text{\textbf{a[0]}}]\!]^{}_{\left\langle{}\left\langle{}X''',\left\langle{}\left(e, Y\right),t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X''', \left(e, Y\right)\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X''',\left\langle{}\left(\left(e, Y\right), Y''\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X''',\left\langle{}\left(e, Y\right),t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X''', \left(e, Y\right)\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X'''} \: . \: \lambda{} o_{\left(\left(e, Y\right), Y''\right)} \: . \: {n}({i})({o}[\textsf{0}]) \wedge{} {v}(({o}[\textsf{0}])[\textsf{0}])({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$
<br />$[\![\text{\textbf{a[1]}}]\!]^{}_{\left\langle{}\left\langle{}X''',\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X''', e\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X''',\left\langle{}\left(e, Y''\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X''',\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X''', e\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X'''} \: . \: \lambda{} o_{\left(e, Y''\right)} \: . \: {n}({i})({o}[\textsf{0}]) \wedge{} {v}({o}[\textsf{0}])({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$
<br />$[\![\text{\textbf{a[2]}}]\!]^{}_{\left\langle{}\left\langle{}X_{4},\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X_{4}, e\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X_{4},\left\langle{}\left(e, Y''\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X_{4},\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{4}, e\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X_{4}} \: . \: \lambda{} o_{\left(e, Y''\right)} \: . \: {n}({i})({o}[\textsf{0}]) \wedge{} {v}({o}[\textsf{0}])({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$<br />
$[\![\text{\textbf{who[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{4}, \left(e, Y'''\right)\right),\left\langle{}Y_{4},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}X_{4},\left\langle{}\left(e, Y'''\right),t\right\rangle{}\right\rangle{},\left\langle{}X_{4},\left\langle{}\left(\left(e, Y'''\right), Y_{4}\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{4}, \left(e, Y'''\right)\right),\left\langle{}Y_{4},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} n_{\left\langle{}X_{4},\left\langle{}\left(e, Y'''\right),t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X_{4}} \: . \: \lambda{} o_{\left(\left(e, Y'''\right), Y_{4}\right)} \: . \: {n}({i})({o}[\textsf{0}]) \wedge{} {v}(({o}[\textsf{0}])[\textsf{0}])({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$
<br />$[\![\text{\textbf{who[1]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{4}, e\right),\left\langle{}Y_{4},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}X_{4},\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}X_{4},\left\langle{}\left(e, Y_{4}\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{4}, e\right),\left\langle{}Y_{4},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} n_{\left\langle{}X_{4},\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X_{4}} \: . \: \lambda{} o_{\left(e, Y_{4}\right)} \: . \: {n}({i})({o}[\textsf{0}]) \wedge{} {v}({o}[\textsf{0}])({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$<br />
$[\![\text{\textbf{E}}]\!]^{}_{\left\langle{}\left\langle{}\left\{e\right\},\left\langle{}Y_{4},t\right\rangle{}\right\rangle{},t\right\rangle{}} \:=\: \lambda{} p_{\left\langle{}\left\{e\right\},\left\langle{}Y_{4},t\right\rangle{}\right\rangle{}} \: . \: \exists{} m_{Y_{4}} \: . \: {p}(\{\}_{\left\{e\right\}})({m})$<br />
$[\![\text{\textbf{C}}]\!]^{}_{\left\langle{}t,t\right\rangle{}} \:=\: \lambda{} p_{t} \: . \: {p}$<br />
${det0}_{\left\langle{}\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{},\left\langle{}\left\langle{}X''',\left\langle{}\left(e, Y\right),t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X''', \left(e, Y\right)\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X''',\left\langle{}\left\langle{}\left(e, Y\right),Y''\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}}\:=\:\lambda{} d_{\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{}} \: . \: \lambda{} n_{\left\langle{}X''',\left\langle{}\left(e, Y\right),t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X''', \left(e, Y\right)\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X'''} \: . \: \lambda{} f_{\left\langle{}\left(e, Y\right),Y''\right\rangle{}} \: . \: (Dom({f}) = (Dom({f}) \cap{} \{{o}_{\left(e, Y\right)} \:|\: {n}({i})({o}_{\left(e, Y\right)})\})) \wedge{} {d}(\{{x}_{e} \:|\: \exists{} y_{Y} \: . \: {n}({i})({x}_{e}, {y})\}, \{{x}_{e} \:|\: \exists{} y_{Y} \: . \: ({x}_{e}, {y}) \in{} Dom({f})\}) \wedge{} (\forall{} o_{\left(e, Y\right)} \: . \: ({o} \in{} Dom({f})) \rightarrow{} {v}({o}[\textsf{0}])({i}, {o})({f}({o})))$<br />
${det1}_{\left\langle{}\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{},\left\langle{}\left\langle{}X''',\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X''', e\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X''',\left\langle{}\left\langle{}e,Y''\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}}\:=\:\lambda{} d_{\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{}} \: . \: \lambda{} n_{\left\langle{}X''',\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X''', e\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X'''} \: . \: \lambda{} f_{\left\langle{}e,Y''\right\rangle{}} \: . \: (Dom({f}) = (Dom({f}) \cap{} \{{z}_{e} \:|\: {n}({i})({z}_{e})\})) \wedge{} {d}(\{{x}_{e} \:|\: {n}({i})({x}_{e})\}, Dom({f})) \wedge{} (\forall{} z_{e} \: . \: ({z} \in{} Dom({f})) \rightarrow{} {v}({z})({i}, {z})({f}({z})))$<br />
$[\![\text{\textbf{every[0]}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}\left(e, X'\right),t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X, \left(e, X'\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}\left(e, X'\right),X''\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X,\left\langle{}\left(e, X'\right),t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X, \left(e, X'\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}\left(e, X'\right),X''\right\rangle{}} \: . \: (\forall{} x_{e} \: . \: (\exists{} y_{X'} \: . \: {n}({i})({x}, {y})) \rightarrow{} (\exists{} y_{X'} \: . \: ({x}, {y}) \in{} Dom({f}))) \wedge{} (Dom({f}) = (Dom({f}) \cap{} \{{o}_{\left(e, X'\right)} \:|\: {n}({i})({o}_{\left(e, X'\right)})\})) \wedge{} (\forall{} o_{\left(e, X'\right)} \: . \: ({o} \in{} Dom({f})) \rightarrow{} {v}({o}[\textsf{0}])({i}, {o})({f}({o})))$
<br />$[\![\text{\textbf{every[1]}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X, e\right),\left\langle{}X',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}e,X'\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X, e\right),\left\langle{}X',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}e,X'\right\rangle{}} \: . \: (\{{x}_{e} \:|\: {n}({i})({x}_{e})\} = (Dom({f}) \cap{} \{{x}_{e} \:|\: {n}({i})({x}_{e})\})) \wedge{} (Dom({f}) = (Dom({f}) \cap{} \{{z}_{e} \:|\: {n}({i})({z}_{e})\})) \wedge{} (\forall{} z_{e} \: . \: ({z} \in{} Dom({f})) \rightarrow{} {v}({z})({i}, {z})({f}({z})))$<br />
$[\![\text{\textbf{most[0]}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}\left(e, X'\right),t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X, \left(e, X'\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}\left(e, X'\right),X''\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X,\left\langle{}\left(e, X'\right),t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X, \left(e, X'\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}\left(e, X'\right),X''\right\rangle{}} \: . \: (Dom({f}) = (Dom({f}) \cap{} \{{o}_{\left(e, X'\right)} \:|\: {n}({i})({o}_{\left(e, X'\right)})\})) \wedge{} ((\textsf{2} * {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: (\exists{} y_{X'} \: . \: ({x}_{e}, {y}) \in{} Dom({f})) \wedge{} (\exists{} y_{X'} \: . \: {n}({i})({x}_{e}, {y}))\})) > {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: \exists{} y_{X'} \: . \: {n}({i})({x}_{e}, {y})\})) \wedge{} (\forall{} o_{\left(e, X'\right)} \: . \: ({o} \in{} Dom({f})) \rightarrow{} {v}({o}[\textsf{0}])({i}, {o})({f}({o})))$
<br />$[\![\text{\textbf{most[1]}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X, e\right),\left\langle{}X',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}e,X'\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X, e\right),\left\langle{}X',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}e,X'\right\rangle{}} \: . \: (Dom({f}) = (Dom({f}) \cap{} \{{z}_{e} \:|\: {n}({i})({z}_{e})\})) \wedge{} ((\textsf{2} * {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: {n}({i})({x}_{e})\} \cap{} Dom({f}))) > {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: {n}({i})({x}_{e})\})) \wedge{} (\forall{} z_{e} \: . \: ({z} \in{} Dom({f})) \rightarrow{} {v}({z})({i}, {z})({f}({z})))$<br />
$[\![\text{\textbf{two[0]}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}\left(e, X'\right),t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X, \left(e, X'\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}\left(e, X'\right),X''\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X,\left\langle{}\left(e, X'\right),t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X, \left(e, X'\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}\left(e, X'\right),X''\right\rangle{}} \: . \: (Dom({f}) = (Dom({f}) \cap{} \{{o}_{\left(e, X'\right)} \:|\: {n}({i})({o}_{\left(e, X'\right)})\})) \wedge{} (\textsf{2} = {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: (\exists{} y_{X'} \: . \: ({x}_{e}, {y}) \in{} Dom({f})) \wedge{} (\exists{} y_{X'} \: . \: {n}({i})({x}_{e}, {y}))\})) \wedge{} (\forall{} o_{\left(e, X'\right)} \: . \: ({o} \in{} Dom({f})) \rightarrow{} {v}({o}[\textsf{0}])({i}, {o})({f}({o})))$
<br />$[\![\text{\textbf{two[1]}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X, e\right),\left\langle{}X',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}e,X'\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X, e\right),\left\langle{}X',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}e,X'\right\rangle{}} \: . \: (Dom({f}) = (Dom({f}) \cap{} \{{z}_{e} \:|\: {n}({i})({z}_{e})\})) \wedge{} (\textsf{2} = {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(\{{x}_{e} \:|\: {n}({i})({x}_{e})\} \cap{} Dom({f}))) \wedge{} (\forall{} z_{e} \: . \: ({z} \in{} Dom({f})) \rightarrow{} {v}({z})({i}, {z})({f}({z})))$<br />
$[\![\text{\textbf{it[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(e, X_{5}\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(e, X_{5}\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, X_{5}\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(e, X_{5}\right)} \: . \: {v}({i}[\textsf{0}])({i})$
<br />$[\![\text{\textbf{it[1]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{5}, e\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{5}, e\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{5}, e\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{5}, e\right)} \: . \: {v}({i}[\textsf{1}])({i})$
<br />$[\![\text{\textbf{it[2]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{5}, \left(e, Y_{5}\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{5}, \left(e, Y_{5}\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{5}, \left(e, Y_{5}\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{5}, \left(e, Y_{5}\right)\right)} \: . \: {v}(({i}[\textsf{1}])[\textsf{0}])({i})$
<br />$[\![\text{\textbf{it[3]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{5}, \left(Y_{5}, e\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{5}, \left(Y_{5}, e\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{5}, \left(Y_{5}, e\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{5}, \left(Y_{5}, e\right)\right)} \: . \: {v}(({i}[\textsf{1}])[\textsf{1}])({i})$
<br />$[\![\text{\textbf{it[4]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{5}, \left(Y_{5}, \left(e, Z_{8}\right)\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{5}, \left(Y_{5}, \left(e, Z_{8}\right)\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{5}, \left(Y_{5}, \left(e, Z_{8}\right)\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{5}, \left(Y_{5}, \left(e, Z_{8}\right)\right)\right)} \: . \: {v}((({i}[\textsf{1}])[\textsf{1}])[\textsf{0}])({i})$
<br />$[\![\text{\textbf{it[5]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{5}, \left(Y_{5}, \left(Z_{8}, e\right)\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{5}, \left(Y_{5}, \left(Z_{8}, e\right)\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{5}, \left(Y_{5}, \left(Z_{8}, e\right)\right)\right),\left\langle{}Z_{7},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{5}, \left(Y_{5}, \left(Z_{8}, e\right)\right)\right)} \: . \: {v}((({i}[\textsf{1}])[\textsf{1}])[\textsf{1}])({i})$
<br />$[\![\text{\textbf{it[6]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(\left\langle{}e,\left(e, Y_{7}\right)\right\rangle{}, e\right),\left\langle{}Z_{6},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(\left\langle{}e,\left(e, Y_{7}\right)\right\rangle{}, e\right),\left\langle{}Z_{6},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(\left\langle{}e,\left(e, Y_{7}\right)\right\rangle{}, e\right),\left\langle{}Z_{6},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(\left\langle{}e,\left(e, Y_{7}\right)\right\rangle{}, e\right)} \: . \: {v}(({i}[\textsf{0}])({i}[\textsf{1}])[\textsf{0}])({i})$
<br />$[\![\text{\textbf{it[7]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(\left(X_{9}, \left\langle{}e,\left(e, Y_{7}\right)\right\rangle{}\right), e\right),\left\langle{}Z_{6},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(\left(X_{9}, \left\langle{}e,\left(e, Y_{7}\right)\right\rangle{}\right), e\right),\left\langle{}Z_{6},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(\left(X_{9}, \left\langle{}e,\left(e, Y_{7}\right)\right\rangle{}\right), e\right),\left\langle{}Z_{6},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(\left(X_{9}, \left\langle{}e,\left(e, Y_{7}\right)\right\rangle{}\right), e\right)} \: . \: {v}((({i}[\textsf{0}])[\textsf{1}])({i}[\textsf{1}])[\textsf{0}])({i})$
<br />$[\![\text{\textbf{it[8]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X, \left(e, \left\langle{}e,\left(e, Z\right)\right\rangle{}\right)\right),\left\langle{}Y,t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X, \left(e, \left\langle{}e,\left(e, Z\right)\right\rangle{}\right)\right),\left\langle{}Y,t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X, \left(e, \left\langle{}e,\left(e, Z\right)\right\rangle{}\right)\right),\left\langle{}Y,t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X, \left(e, \left\langle{}e,\left(e, Z\right)\right\rangle{}\right)\right)} \: . \: {v}((({i}[\textsf{1}])[\textsf{1}])(({i}[\textsf{1}])[\textsf{0}])[\textsf{0}])({i})$<br />
$[\![\text{\textbf{he[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(e, X_{6}\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(e, X_{6}\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(e, X_{6}\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(e, X_{6}\right)} \: . \: {v}({i}[\textsf{0}])({i})$
<br />$[\![\text{\textbf{he[1]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{6}, e\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{6}, e\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{6}, e\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{6}, e\right)} \: . \: {v}({i}[\textsf{1}])({i})$
<br />$[\![\text{\textbf{he[2]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{6}, \left(e, Y_{6}\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{6}, \left(e, Y_{6}\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{6}, \left(e, Y_{6}\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{6}, \left(e, Y_{6}\right)\right)} \: . \: {v}(({i}[\textsf{1}])[\textsf{0}])({i})$
<br />$[\![\text{\textbf{he[3]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{6}, \left(Y_{6}, e\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{6}, \left(Y_{6}, e\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{6}, \left(Y_{6}, e\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{6}, \left(Y_{6}, e\right)\right)} \: . \: {v}(({i}[\textsf{1}])[\textsf{1}])({i})$
<br />$[\![\text{\textbf{he[4]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{6}, \left(Y_{6}, \left(e, Z_{9}\right)\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{6}, \left(Y_{6}, \left(e, Z_{9}\right)\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{6}, \left(Y_{6}, \left(e, Z_{9}\right)\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{6}, \left(Y_{6}, \left(e, Z_{9}\right)\right)\right)} \: . \: {v}((({i}[\textsf{1}])[\textsf{1}])[\textsf{0}])({i})$
<br />$[\![\text{\textbf{he[5]}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{6}, \left(Y_{6}, \left(Z_{9}, e\right)\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}\left(X_{6}, \left(Y_{6}, \left(Z_{9}, e\right)\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{6}, \left(Y_{6}, \left(Z_{9}, e\right)\right)\right),\left\langle{}Z_{8},t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X_{6}, \left(Y_{6}, \left(Z_{9}, e\right)\right)\right)} \: . \: {v}((({i}[\textsf{1}])[\textsf{1}])[\textsf{1}])({i})$<br />
$[\![\text{\textbf{aND}}]\!]^{}_{\left\langle{}\left\langle{}\left(X_{9}, Y_{9}\right),\left\langle{}Z_{9},t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}X_{9},\left\langle{}Y_{9},t\right\rangle{}\right\rangle{},\left\langle{}X_{9},\left\langle{}\left(Y_{9}, Z_{9}\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} q_{\left\langle{}\left(X_{9}, Y_{9}\right),\left\langle{}Z_{9},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} p_{\left\langle{}X_{9},\left\langle{}Y_{9},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X_{9}} \: . \: \lambda{} o_{\left(Y_{9}, Z_{9}\right)} \: . \: {p}({i})({o}[\textsf{0}]) \wedge{} {q}({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$<br />
$[\![\text{\textbf{iF}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}Y,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}\left(X, Y\right),\left\langle{}Z,t\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}Y,Z\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} p_{\left\langle{}X,\left\langle{}Y,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} q_{\left\langle{}\left(X, Y\right),\left\langle{}Z,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}Y,Z\right\rangle{}} \: . \: (Dom({f}) = \{{a}_{Y} \:|\: {p}({i})({a}_{Y})\}) \wedge{} (\forall{} a_{Y} \: . \: ({a} \in{} Dom({f})) \rightarrow{} {q}({i}, {a})({f}({a})))$<br />
$[\![\text{\textbf{of\_them[0]}}]\!]^{}_{\left\langle{}\left\langle{}e,X_{7}\right\rangle{},\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{\left\langle{}e,X_{7}\right\rangle{}} \: . \: \lambda{} x_{e} \: . \: {x} \in{} Dom({i})$
<br />$[\![\text{\textbf{of\_them[1]}}]\!]^{}_{\left\langle{}\left(X_{9}, \left\langle{}e,X_{7}\right\rangle{}\right),\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{\left(X_{9}, \left\langle{}e,X_{7}\right\rangle{}\right)} \: . \: \lambda{} x_{e} \: . \: {x} \in{} Dom({i}[\textsf{1}])$

In [5]:
# Traces and binders
t = [lang.Trace(n) for n in range(5)]
b = [lang.Binder(n) for n in range(5)]

In [6]:
counter = 0
def eg(example:str, counter:int) -> None:
    print(f'({counter})\t{example}\n')

def exemplify(example:str, counter:int, compResult) -> None:
    eg(example, counter)
    if len(compResult)<1:
        raise Exception("Composition failed for some reason.")
    print(compResult.source)
    print("")
    if len(compResult)<2:
        display(compResult.content[0].content)
    else:
        for n,cont in enumerate(compResult.content):
            print(f"Result [{n}]:")
            display(cont.content)

## Examples

### Simple Sentences

In [7]:
counter += 1
exemplify("A donkey brays.", counter, (C * (E * ((a * donkey) * brays))))

(1)	A donkey brays.

[C [E [[a donkey] brays]]]



(Exists m_(e,v): (Bray_<(e,v),t>((m_(e,v)[0]), (m_(e,v)[1])) & Donkey_<e,t>(m_(e,v)[0])))

In [8]:
counter += 1
exemplify("Every donkey brays.", counter, (C * (E * ((every * donkey) * brays))))

(2)	Every donkey brays.

[C [E [[every donkey] brays]]]



(Exists m_<e,v>: ((((Set x_e: Donkey_<e,t>(x_e)) <=> (Dom(m_<e,v>) & (Set x_e: Donkey_<e,t>(x_e)))) & (Dom(m_<e,v>) <=> (Dom(m_<e,v>) & (Set z_e: Donkey_<e,t>(z_e))))) & (Forall z_e: ((z_e << Dom(m_<e,v>)) >> Bray_<(e,v),t>(z_e, m_<e,v>(z_e))))))

In [9]:
counter += 1
exemplify("Most donkeys bray.", counter, (C * (E * ((most * donkeys) * bray))))

(3)	Most donkeys bray.

[C [E [[most donkeys] bray]]]



(Exists m_<e,v>: (((Dom(m_<e,v>) <=> (Dom(m_<e,v>) & (Set z_e: Donkey_<e,t>(z_e)))) & ((2 * Card_<{e},n>(Dom(m_<e,v>) & (Set x_e: Donkey_<e,t>(x_e)))) > Card_<{e},n>(Set x_e: Donkey_<e,t>(x_e)))) & (Forall z_e: ((z_e << Dom(m_<e,v>)) >> Bray_<(e,v),t>(z_e, m_<e,v>(z_e))))))

In [10]:
counter += 1
exemplify("A farmer owns a donkey.", counter, (C * (E * ((a * farmer) * (b[2] * ((a * donkey) * (b[1] * (t[2] * (owns * t[1])))))))))

(4)	A farmer owns a donkey.

[C [E [[a farmer] [2 [[a donkey] [1 [t2 [owns t1]]]]]]]]



(Exists m_(e,(e,v)): ((Donkey_<e,t>((m_(e,(e,v))[1])[0]) & Farmer_<e,t>(m_(e,(e,v))[0])) & Own_<(e,e,v),t>((m_(e,(e,v))[0]), ((m_(e,(e,v))[1])[0]), ((m_(e,(e,v))[1])[1]))))

In [11]:
counter += 1
exemplify("Every farmer owns a donkey.", counter, (C * (E * ((every * farmer) * (b[2] * ((a * donkey) * (b[1] * (t[2] * (owns * t[1])))))))))

(5)	Every farmer owns a donkey.

[C [E [[every farmer] [2 [[a donkey] [1 [t2 [owns t1]]]]]]]]



(Exists m_<e,(e,v)>: ((((Set x_e: Farmer_<e,t>(x_e)) <=> (Dom(m_<e,(e,v)>) & (Set x_e: Farmer_<e,t>(x_e)))) & (Dom(m_<e,(e,v)>) <=> (Dom(m_<e,(e,v)>) & (Set z_e: Farmer_<e,t>(z_e))))) & (Forall z_e: ((z_e << Dom(m_<e,(e,v)>)) >> (Donkey_<e,t>(m_<e,(e,v)>(z_e)[0]) & Own_<(e,e,v),t>(z_e, (m_<e,(e,v)>(z_e)[0]), (m_<e,(e,v)>(z_e)[1])))))))

###  Dynamic Binding

In [12]:
counter += 1
exemplify("Every farmer who owns a donkey feeds it.", counter, ((C * (E  * ((every * (farmer * (who * (b[2] * ((a * donkey) * (b[1] * (t[2] * (owns * t[1])))))))) * (b[4] * (it * (b[3] * (t[4] * (feeds * t[3]))))))))))
print('''\nResult [1] is the intended interpretation. Result [0] has "it" anaphoric on "farmer"; this system is not set up out-of-the-box
to exclude this reading.\n''')

(6)	Every farmer who owns a donkey feeds it.

[C [E [[every [farmer [who [2 [[a donkey] [1 [t2 [owns t1]]]]]]]] [4 [it [3 [t4 [feeds t3]]]]]]]]

Result [0]:


(Exists m_<(e,(e,v)),v>: (((Dom(m_<(e,(e,v)),v>) <=> (Dom(m_<(e,(e,v)),v>) & (Set o_(e,(e,v)): ((Donkey_<e,t>((o_(e,(e,v))[1])[0]) & Farmer_<e,t>(o_(e,(e,v))[0])) & Own_<(e,e,v),t>((o_(e,(e,v))[0]), ((o_(e,(e,v))[1])[0]), ((o_(e,(e,v))[1])[1])))))) & (Forall o_(e,(e,v)): ((o_(e,(e,v)) << Dom(m_<(e,(e,v)),v>)) >> Feed_<(e,e,v),t>((o_(e,(e,v))[0]), (o_(e,(e,v))[0]), m_<(e,(e,v)),v>(o_(e,(e,v))))))) & (Forall x_e: ((Exists y_(e,v): ((Donkey_<e,t>(y_(e,v)[0]) & Farmer_<e,t>(x_e)) & Own_<(e,e,v),t>(x_e, (y_(e,v)[0]), (y_(e,v)[1])))) >> (Exists y_(e,v): ((x_e, y_(e,v)) << Dom(m_<(e,(e,v)),v>)))))))

Result [1]:


(Exists m_<(e,(e,v)),v>: (((Dom(m_<(e,(e,v)),v>) <=> (Dom(m_<(e,(e,v)),v>) & (Set o_(e,(e,v)): ((Donkey_<e,t>((o_(e,(e,v))[1])[0]) & Farmer_<e,t>(o_(e,(e,v))[0])) & Own_<(e,e,v),t>((o_(e,(e,v))[0]), ((o_(e,(e,v))[1])[0]), ((o_(e,(e,v))[1])[1])))))) & (Forall o_(e,(e,v)): ((o_(e,(e,v)) << Dom(m_<(e,(e,v)),v>)) >> Feed_<(e,e,v),t>((o_(e,(e,v))[0]), ((o_(e,(e,v))[1])[0]), m_<(e,(e,v)),v>(o_(e,(e,v))))))) & (Forall x_e: ((Exists y_(e,v): ((Donkey_<e,t>(y_(e,v)[0]) & Farmer_<e,t>(x_e)) & Own_<(e,e,v),t>(x_e, (y_(e,v)[0]), (y_(e,v)[1])))) >> (Exists y_(e,v): ((x_e, y_(e,v)) << Dom(m_<(e,(e,v)),v>)))))))


Result [1] is the intended interpretation. Result [0] has "it" anaphoric on "farmer"; this system is not set up out-of-the-box
to exclude this reading.



In [13]:
counter += 1
exemplify("Most farmers who own a donkey feed it.", counter, (C * (E  * ((most * (farmers * (who * (b[2] * ((a * donkey) * (b[1] * (t[2] * (own * t[1])))))))) * (b[4] * (it * (b[3] * (t[4] * (feed * t[3])))))))))
print("\nResult [1] is the intended interpretation.\n")

(7)	Most farmers who own a donkey feed it.

[C [E [[most [farmers [who [2 [[a donkey] [1 [t2 [own t1]]]]]]]] [4 [it [3 [t4 [feed t3]]]]]]]]

Result [0]:


(Exists m_<(e,(e,v)),v>: (((Dom(m_<(e,(e,v)),v>) <=> (Dom(m_<(e,(e,v)),v>) & (Set o_(e,(e,v)): ((Donkey_<e,t>((o_(e,(e,v))[1])[0]) & Farmer_<e,t>(o_(e,(e,v))[0])) & Own_<(e,e,v),t>((o_(e,(e,v))[0]), ((o_(e,(e,v))[1])[0]), ((o_(e,(e,v))[1])[1])))))) & ((2 * Card_<{e},n>(Set x_e: ((Exists y_(e,v): ((Donkey_<e,t>(y_(e,v)[0]) & Farmer_<e,t>(x_e)) & Own_<(e,e,v),t>(x_e, (y_(e,v)[0]), (y_(e,v)[1])))) & (Exists y_(e,v): ((x_e, y_(e,v)) << Dom(m_<(e,(e,v)),v>)))))) > Card_<{e},n>(Set x_e: (Exists y_(e,v): ((Donkey_<e,t>(y_(e,v)[0]) & Farmer_<e,t>(x_e)) & Own_<(e,e,v),t>(x_e, (y_(e,v)[0]), (y_(e,v)[1]))))))) & (Forall o_(e,(e,v)): ((o_(e,(e,v)) << Dom(m_<(e,(e,v)),v>)) >> Feed_<(e,e,v),t>((o_(e,(e,v))[0]), (o_(e,(e,v))[0]), m_<(e,(e,v)),v>(o_(e,(e,v))))))))

Result [1]:


(Exists m_<(e,(e,v)),v>: (((Dom(m_<(e,(e,v)),v>) <=> (Dom(m_<(e,(e,v)),v>) & (Set o_(e,(e,v)): ((Donkey_<e,t>((o_(e,(e,v))[1])[0]) & Farmer_<e,t>(o_(e,(e,v))[0])) & Own_<(e,e,v),t>((o_(e,(e,v))[0]), ((o_(e,(e,v))[1])[0]), ((o_(e,(e,v))[1])[1])))))) & ((2 * Card_<{e},n>(Set x_e: ((Exists y_(e,v): ((Donkey_<e,t>(y_(e,v)[0]) & Farmer_<e,t>(x_e)) & Own_<(e,e,v),t>(x_e, (y_(e,v)[0]), (y_(e,v)[1])))) & (Exists y_(e,v): ((x_e, y_(e,v)) << Dom(m_<(e,(e,v)),v>)))))) > Card_<{e},n>(Set x_e: (Exists y_(e,v): ((Donkey_<e,t>(y_(e,v)[0]) & Farmer_<e,t>(x_e)) & Own_<(e,e,v),t>(x_e, (y_(e,v)[0]), (y_(e,v)[1]))))))) & (Forall o_(e,(e,v)): ((o_(e,(e,v)) << Dom(m_<(e,(e,v)),v>)) >> Feed_<(e,e,v),t>((o_(e,(e,v))[0]), ((o_(e,(e,v))[1])[0]), m_<(e,(e,v)),v>(o_(e,(e,v))))))))


Result [1] is the intended interpretation.



In [14]:
counter += 1
exemplify("If a farmer owns a donkey, he feeds it.", counter, (C * (E * ((iF * ((a * farmer) * (b[2] * ((a * donkey) * (b[1] * (t[2] * (owns * t[1]))))))) * (he * (b[4] * (it * (b[3] * (t[4] * (feeds * t[3]))))))))))
print('\nResult [1] is the intended interpretation, where "he" is anaphoric on "farmer" and "it" is anaphoric on "donkey".\n')

(8)	If a farmer owns a donkey, he feeds it.

[C [E [[iF [[a farmer] [2 [[a donkey] [1 [t2 [owns t1]]]]]]] [he [4 [it [3 [t4 [feeds t3]]]]]]]]]

Result [0]:


(Exists m_<(e,(e,v)),v>: ((Dom(m_<(e,(e,v)),v>) <=> (Set a_(e,(e,v)): ((Donkey_<e,t>((a_(e,(e,v))[1])[0]) & Farmer_<e,t>(a_(e,(e,v))[0])) & Own_<(e,e,v),t>((a_(e,(e,v))[0]), ((a_(e,(e,v))[1])[0]), ((a_(e,(e,v))[1])[1]))))) & (Forall a_(e,(e,v)): ((a_(e,(e,v)) << Dom(m_<(e,(e,v)),v>)) >> Feed_<(e,e,v),t>((a_(e,(e,v))[0]), (a_(e,(e,v))[0]), m_<(e,(e,v)),v>(a_(e,(e,v))))))))

Result [1]:


(Exists m_<(e,(e,v)),v>: ((Dom(m_<(e,(e,v)),v>) <=> (Set a_(e,(e,v)): ((Donkey_<e,t>((a_(e,(e,v))[1])[0]) & Farmer_<e,t>(a_(e,(e,v))[0])) & Own_<(e,e,v),t>((a_(e,(e,v))[0]), ((a_(e,(e,v))[1])[0]), ((a_(e,(e,v))[1])[1]))))) & (Forall a_(e,(e,v)): ((a_(e,(e,v)) << Dom(m_<(e,(e,v)),v>)) >> Feed_<(e,e,v),t>((a_(e,(e,v))[0]), ((a_(e,(e,v))[1])[0]), m_<(e,(e,v)),v>(a_(e,(e,v))))))))

Result [2]:


(Exists m_<(e,(e,v)),v>: ((Dom(m_<(e,(e,v)),v>) <=> (Set a_(e,(e,v)): ((Donkey_<e,t>((a_(e,(e,v))[1])[0]) & Farmer_<e,t>(a_(e,(e,v))[0])) & Own_<(e,e,v),t>((a_(e,(e,v))[0]), ((a_(e,(e,v))[1])[0]), ((a_(e,(e,v))[1])[1]))))) & (Forall a_(e,(e,v)): ((a_(e,(e,v)) << Dom(m_<(e,(e,v)),v>)) >> Feed_<(e,e,v),t>(((a_(e,(e,v))[1])[0]), (a_(e,(e,v))[0]), m_<(e,(e,v)),v>(a_(e,(e,v))))))))

Result [3]:


(Exists m_<(e,(e,v)),v>: ((Dom(m_<(e,(e,v)),v>) <=> (Set a_(e,(e,v)): ((Donkey_<e,t>((a_(e,(e,v))[1])[0]) & Farmer_<e,t>(a_(e,(e,v))[0])) & Own_<(e,e,v),t>((a_(e,(e,v))[0]), ((a_(e,(e,v))[1])[0]), ((a_(e,(e,v))[1])[1]))))) & (Forall a_(e,(e,v)): ((a_(e,(e,v)) << Dom(m_<(e,(e,v)),v>)) >> Feed_<(e,e,v),t>(((a_(e,(e,v))[1])[0]), ((a_(e,(e,v))[1])[0]), m_<(e,(e,v)),v>(a_(e,(e,v))))))))


Result [1] is the intended interpretation, where "he" is anaphoric on "farmer" and "it" is anaphoric on "donkey".



In [15]:
counter += 1
exemplify("A farmer owns a donkey, and he feeds it.", counter, (C * (E * (((a * farmer) * (b[2] * ((a * donkey) * (b[1] * (t[2] * (owns * t[1])))))) * (aND * (he * (b[4] * (it * (b[3] * (t[4] * (feeds * t[3])))))))))))
print("\nResult [1] is the intended interpretation.\n")

(9)	A farmer owns a donkey, and he feeds it.

[C [E [[[a farmer] [2 [[a donkey] [1 [t2 [owns t1]]]]]] [aND [he [4 [it [3 [t4 [feeds t3]]]]]]]]]]

Result [0]:


(Exists m_((e,(e,v)),v): (((Donkey_<e,t>(((m_((e,(e,v)),v)[0])[1])[0]) & Farmer_<e,t>((m_((e,(e,v)),v)[0])[0])) & Feed_<(e,e,v),t>(((m_((e,(e,v)),v)[0])[0]), ((m_((e,(e,v)),v)[0])[0]), (m_((e,(e,v)),v)[1]))) & Own_<(e,e,v),t>(((m_((e,(e,v)),v)[0])[0]), (((m_((e,(e,v)),v)[0])[1])[0]), (((m_((e,(e,v)),v)[0])[1])[1]))))

Result [1]:


(Exists m_((e,(e,v)),v): (((Donkey_<e,t>(((m_((e,(e,v)),v)[0])[1])[0]) & Farmer_<e,t>((m_((e,(e,v)),v)[0])[0])) & Feed_<(e,e,v),t>(((m_((e,(e,v)),v)[0])[0]), (((m_((e,(e,v)),v)[0])[1])[0]), (m_((e,(e,v)),v)[1]))) & Own_<(e,e,v),t>(((m_((e,(e,v)),v)[0])[0]), (((m_((e,(e,v)),v)[0])[1])[0]), (((m_((e,(e,v)),v)[0])[1])[1]))))

Result [2]:


(Exists m_((e,(e,v)),v): (((Donkey_<e,t>(((m_((e,(e,v)),v)[0])[1])[0]) & Farmer_<e,t>((m_((e,(e,v)),v)[0])[0])) & Feed_<(e,e,v),t>((((m_((e,(e,v)),v)[0])[1])[0]), ((m_((e,(e,v)),v)[0])[0]), (m_((e,(e,v)),v)[1]))) & Own_<(e,e,v),t>(((m_((e,(e,v)),v)[0])[0]), (((m_((e,(e,v)),v)[0])[1])[0]), (((m_((e,(e,v)),v)[0])[1])[1]))))

Result [3]:


(Exists m_((e,(e,v)),v): (((Donkey_<e,t>(((m_((e,(e,v)),v)[0])[1])[0]) & Farmer_<e,t>((m_((e,(e,v)),v)[0])[0])) & Feed_<(e,e,v),t>((((m_((e,(e,v)),v)[0])[1])[0]), (((m_((e,(e,v)),v)[0])[1])[0]), (m_((e,(e,v)),v)[1]))) & Own_<(e,e,v),t>(((m_((e,(e,v)),v)[0])[0]), (((m_((e,(e,v)),v)[0])[1])[0]), (((m_((e,(e,v)),v)[0])[1])[1]))))


Result [1] is the intended interpretation.



### Quantificational Subordination

In [16]:
counter += 1
exemplify("Every farmer owns a donkey. Most of them feed it.", counter, (C * (E * (((every * farmer) * (b[2] * ((a * donkey) * (b[1] * (t[2] * (owns * t[1])))))) * (aND * ((most * of_them) * (b[4] * (it * (b[3] * (t[4] * (feed * t[3])))))))))))
print("\nResult [1] is the intended interpretation.\n")

(10)	Every farmer owns a donkey. Most of them feed it.

[C [E [[[every farmer] [2 [[a donkey] [1 [t2 [owns t1]]]]]] [aND [[most of_them] [4 [it [3 [t4 [feed t3]]]]]]]]]]

Result [0]:


(Exists m_(<e,(e,v)>,<e,v>): (((((((Set x_e: Farmer_<e,t>(x_e)) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set x_e: Farmer_<e,t>(x_e)))) & (Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set z_e: Farmer_<e,t>(z_e))))) & (Dom(m_(<e,(e,v)>,<e,v>)[1]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & ((2 * Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1]))) > Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0])))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[0])) >> (Donkey_<e,t>((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]) & Own_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[1])))))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[1])) >> Feed_<(e,e,v),t>(z_e, z_e, (m_(<e,(e,v)>,<e,v>)[1])(z_e))))))

Result [1]:


(Exists m_(<e,(e,v)>,<e,v>): (((((((Set x_e: Farmer_<e,t>(x_e)) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set x_e: Farmer_<e,t>(x_e)))) & (Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set z_e: Farmer_<e,t>(z_e))))) & (Dom(m_(<e,(e,v)>,<e,v>)[1]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & ((2 * Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1]))) > Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0])))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[0])) >> (Donkey_<e,t>((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]) & Own_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[1])))))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[1])) >> Feed_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), (m_(<e,(e,v)>,<e,v>)[1])(z_e))))))


Result [1] is the intended interpretation.



In [17]:
counter += 1
exemplify("Most farmers own a donkey. Two of them feed it.", counter, (C * (E * (((most * farmers) * (b[2] * ((a * donkey) * (b[1] * (t[2] * (own * t[1])))))) * (aND * ((two * of_them) * (b[4] * (it * (b[3] * (t[4] * (feed * t[3])))))))))))
print("\nResult [1] is the intended interpretation.\n")

(11)	Most farmers own a donkey. Two of them feed it.

[C [E [[[most farmers] [2 [[a donkey] [1 [t2 [own t1]]]]]] [aND [[two of_them] [4 [it [3 [t4 [feed t3]]]]]]]]]]

Result [0]:


(Exists m_(<e,(e,v)>,<e,v>): ((((((Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set z_e: Farmer_<e,t>(z_e)))) & (Dom(m_(<e,(e,v)>,<e,v>)[1]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & (2 <=> Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & ((2 * Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set x_e: Farmer_<e,t>(x_e)))) > Card_<{e},n>(Set x_e: Farmer_<e,t>(x_e)))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[0])) >> (Donkey_<e,t>((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]) & Own_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[1])))))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[1])) >> Feed_<(e,e,v),t>(z_e, z_e, (m_(<e,(e,v)>,<e,v>)[1])(z_e))))))

Result [1]:


(Exists m_(<e,(e,v)>,<e,v>): ((((((Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set z_e: Farmer_<e,t>(z_e)))) & (Dom(m_(<e,(e,v)>,<e,v>)[1]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & (2 <=> Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & ((2 * Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set x_e: Farmer_<e,t>(x_e)))) > Card_<{e},n>(Set x_e: Farmer_<e,t>(x_e)))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[0])) >> (Donkey_<e,t>((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]) & Own_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[1])))))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[1])) >> Feed_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), (m_(<e,(e,v)>,<e,v>)[1])(z_e))))))


Result [1] is the intended interpretation.



### Telescoping

For exposition, we'll treat subordinating adverbials like _always_ and _usually_ as sentence-level, and for simplicity's sake we'll only look at cases with unmodified noun phrases.

In [18]:
%%lamb
subconj = L d_<({e},{e}),t>: L q_<(X,(e,<e,Y>)),<Z,t>>: L p_<X,<<e,Y>,t>>: L i_X: L u_(<e,Y>,<e,Z>): p(i)(u[0]) & (Dom(u[1]) <= Dom(u[0])) & d(Dom(u[0]),Dom(u[1])) & (Forall x_e: (x << Dom(u[1])) ==> q((i,(x,u[0])))(u[1](x)))
||always|| = subconj(L o_({e},{e}): o[0] <= o[1])
||usually|| = subconj(L o_({e},{e}): 2 * Card_<{e},n>(o[0] & o[1]) > Card_<{e},n>(o[0]))
||player|| = L i_X1: Player_<e,t>
||figure|| = L i_X1: Figure_<e,t>
||chooses|| = L y_e: L x_e: L i_X2: L e_v: Choose_<(e,e,v),t>(x,y,e)
||puts_on_board|| = L y_e: L x_e: L i_X2: L e_v: Put_<(e,e,e,v),t>(x,y,OnBoard_e,e)

${subconj}_{\left\langle{}\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{},\left\langle{}\left\langle{}\left(X, \left(e, \left\langle{}e,Y\right\rangle{}\right)\right),\left\langle{}Z,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}X,\left\langle{}\left\langle{}e,Y\right\rangle{},t\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left(\left\langle{}e,Y\right\rangle{}, \left\langle{}e,Z\right\rangle{}\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}}\:=\:\lambda{} d_{\left\langle{}\left(\left\{e\right\}, \left\{e\right\}\right),t\right\rangle{}} \: . \: \lambda{} q_{\left\langle{}\left(X, \left(e, \left\langle{}e,Y\right\rangle{}\right)\right),\left\langle{}Z,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} p_{\left\langle{}X,\left\langle{}\left\langle{}e,Y\right\rangle{},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} u_{\left(\left\langle{}e,Y\right\rangle{}, \left\langle{}e,Z\right\rangle{}\right)} \: . \: (Dom({u}[\textsf{1}]) = (Dom({u}[\textsf{0}]) \cap{} Dom({u}[\textsf{1}]))) \wedge{} {d}(Dom({u}[\textsf{0}]), Dom({u}[\textsf{1}])) \wedge{} {p}({i})({u}[\textsf{0}]) \wedge{} (\forall{} x_{e} \: . \: ({x} \in{} Dom({u}[\textsf{1}])) \rightarrow{} {q}({i}, ({x}, ({u}[\textsf{0}])))(({u}[\textsf{1}])({x})))$<br />
$[\![\text{\textbf{always}}]\!]^{}_{\left\langle{}\left\langle{}\left(X, \left(e, \left\langle{}e,X'\right\rangle{}\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}X,\left\langle{}\left\langle{}e,X'\right\rangle{},t\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left(\left\langle{}e,X'\right\rangle{}, \left\langle{}e,X''\right\rangle{}\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} q_{\left\langle{}\left(X, \left(e, \left\langle{}e,X'\right\rangle{}\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{}} \: . \: \lambda{} p_{\left\langle{}X,\left\langle{}\left\langle{}e,X'\right\rangle{},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} u_{\left(\left\langle{}e,X'\right\rangle{}, \left\langle{}e,X''\right\rangle{}\right)} \: . \: (Dom({u}[\textsf{0}]) = (Dom({u}[\textsf{0}]) \cap{} Dom({u}[\textsf{1}]))) \wedge{} (Dom({u}[\textsf{1}]) = (Dom({u}[\textsf{0}]) \cap{} Dom({u}[\textsf{1}]))) \wedge{} {p}({i})({u}[\textsf{0}]) \wedge{} (\forall{} x_{e} \: . \: ({x} \in{} Dom({u}[\textsf{1}])) \rightarrow{} {q}({i}, ({x}, ({u}[\textsf{0}])))(({u}[\textsf{1}])({x})))$<br />
$[\![\text{\textbf{usually}}]\!]^{}_{\left\langle{}\left\langle{}\left(X, \left(e, \left\langle{}e,X'\right\rangle{}\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}X,\left\langle{}\left\langle{}e,X'\right\rangle{},t\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left(\left\langle{}e,X'\right\rangle{}, \left\langle{}e,X''\right\rangle{}\right),t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} q_{\left\langle{}\left(X, \left(e, \left\langle{}e,X'\right\rangle{}\right)\right),\left\langle{}X'',t\right\rangle{}\right\rangle{}} \: . \: \lambda{} p_{\left\langle{}X,\left\langle{}\left\langle{}e,X'\right\rangle{},t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} u_{\left(\left\langle{}e,X'\right\rangle{}, \left\langle{}e,X''\right\rangle{}\right)} \: . \: (Dom({u}[\textsf{1}]) = (Dom({u}[\textsf{0}]) \cap{} Dom({u}[\textsf{1}]))) \wedge{} ((\textsf{2} * {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(Dom({u}[\textsf{0}]) \cap{} Dom({u}[\textsf{1}]))) > {Card}_{\left\langle{}\left\{e\right\},n\right\rangle{}}(Dom({u}[\textsf{0}]))) \wedge{} {p}({i})({u}[\textsf{0}]) \wedge{} (\forall{} x_{e} \: . \: ({x} \in{} Dom({u}[\textsf{1}])) \rightarrow{} {q}({i}, ({x}, ({u}[\textsf{0}])))(({u}[\textsf{1}])({x})))$<br />
$[\![\text{\textbf{player}}]\!]^{}_{\left\langle{}X',\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{X'} \: . \: {Player}$<br />
$[\![\text{\textbf{figure}}]\!]^{}_{\left\langle{}X',\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{X'} \: . \: {Figure}$<br />
$[\![\text{\textbf{chooses}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}X'',\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{X''} \: . \: \lambda{} e_{v} \: . \: {Choose}({x}, {y}, {e})$<br />
$[\![\text{\textbf{puts\_on\_board}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}X'',\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{X''} \: . \: \lambda{} e_{v} \: . \: {Put}({x}, {y}, {OnBoard}_{e}, {e})$

We'll treat the example of telescoping with an overt adverbial as arising from a covert _always_.

In [19]:
counter += 1
eg("Every player chooses a figure. He puts it on the board.", counter)
counter += 1
exemplify("Every player chooses a figure. He always puts it on the board.", counter, (C * (E * (((every * player) * (b[2] * ((a * figure) * (b[1] * (t[2] * (chooses * t[1])))))) * (always * (he * (b[4] * (it * (b[3] * (t[4] * (puts_on_board * t[3])))))))))))
print("\nResult [1] is the intended interpretation.\n")

(12)	Every player chooses a figure. He puts it on the board.

(13)	Every player chooses a figure. He always puts it on the board.

[C [E [[[every player] [2 [[a figure] [1 [t2 [chooses t1]]]]]] [always [he [4 [it [3 [t4 [puts_on_board t3]]]]]]]]]]

Result [0]:


(Exists m_(<e,(e,v)>,<e,v>): (((((((Set x_e: Player_<e,t>(x_e)) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set x_e: Player_<e,t>(x_e)))) & (Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set z_e: Player_<e,t>(z_e))))) & (Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & (Dom(m_(<e,(e,v)>,<e,v>)[1]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & (Forall x_e: ((x_e << Dom(m_(<e,(e,v)>,<e,v>)[1])) >> Put_<(e,e,e,v),t>(x_e, x_e, OnBoard_e, (m_(<e,(e,v)>,<e,v>)[1])(x_e))))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[0])) >> (Choose_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[1])) & Figure_<e,t>((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]))))))

Result [1]:


(Exists m_(<e,(e,v)>,<e,v>): (((((((Set x_e: Player_<e,t>(x_e)) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set x_e: Player_<e,t>(x_e)))) & (Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set z_e: Player_<e,t>(z_e))))) & (Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & (Dom(m_(<e,(e,v)>,<e,v>)[1]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & (Forall x_e: ((x_e << Dom(m_(<e,(e,v)>,<e,v>)[1])) >> Put_<(e,e,e,v),t>(x_e, ((m_(<e,(e,v)>,<e,v>)[0])(x_e)[0]), OnBoard_e, (m_(<e,(e,v)>,<e,v>)[1])(x_e))))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[0])) >> (Choose_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[1])) & Figure_<e,t>((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]))))))


Result [1] is the intended interpretation.



In [20]:
counter == 1
exemplify("Every player chooses a figure. He usually puts it on the board.", counter, (C * (E * (((every * player) * (b[2] * ((a * figure) * (b[1] * (t[2] * (owns * t[1])))))) * (usually * (he * (b[4] * (it * (b[3] * (t[4] * (puts_on_board * t[3])))))))))))
print("\nResult [1] is the intended interpretation.\n")

(13)	Every player chooses a figure. He usually puts it on the board.

[C [E [[[every player] [2 [[a figure] [1 [t2 [owns t1]]]]]] [usually [he [4 [it [3 [t4 [puts_on_board t3]]]]]]]]]]

Result [0]:


(Exists m_(<e,(e,v)>,<e,v>): (((((((Set x_e: Player_<e,t>(x_e)) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set x_e: Player_<e,t>(x_e)))) & (Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set z_e: Player_<e,t>(z_e))))) & (Dom(m_(<e,(e,v)>,<e,v>)[1]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & ((2 * Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1]))) > Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0])))) & (Forall x_e: ((x_e << Dom(m_(<e,(e,v)>,<e,v>)[1])) >> Put_<(e,e,e,v),t>(x_e, x_e, OnBoard_e, (m_(<e,(e,v)>,<e,v>)[1])(x_e))))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[0])) >> (Figure_<e,t>((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]) & Own_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[1])))))))

Result [1]:


(Exists m_(<e,(e,v)>,<e,v>): (((((((Set x_e: Player_<e,t>(x_e)) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set x_e: Player_<e,t>(x_e)))) & (Dom(m_(<e,(e,v)>,<e,v>)[0]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & (Set z_e: Player_<e,t>(z_e))))) & (Dom(m_(<e,(e,v)>,<e,v>)[1]) <=> (Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1])))) & ((2 * Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0]) & Dom(m_(<e,(e,v)>,<e,v>)[1]))) > Card_<{e},n>(Dom(m_(<e,(e,v)>,<e,v>)[0])))) & (Forall x_e: ((x_e << Dom(m_(<e,(e,v)>,<e,v>)[1])) >> Put_<(e,e,e,v),t>(x_e, ((m_(<e,(e,v)>,<e,v>)[0])(x_e)[0]), OnBoard_e, (m_(<e,(e,v)>,<e,v>)[1])(x_e))))) & (Forall z_e: ((z_e << Dom(m_(<e,(e,v)>,<e,v>)[0])) >> (Figure_<e,t>((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]) & Own_<(e,e,v),t>(z_e, ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[0]), ((m_(<e,(e,v)>,<e,v>)[0])(z_e)[1])))))))


Result [1] is the intended interpretation.



## An Extension: _different_

The following ideas are not found in the aforementioned papers, but I hope to show that the system developed there is naturally extendable to account for them.

- Sentence-external _different_, e.g. &lsquo;Bob read _War and Peace_. Fred read a different book.&rsquo; (different to _War and Peace_).
- Sentence-internal _different_, e.g. &lsquo;Every student read a different book.&rsquo; (different to each other).

### Sentence-external

For sentence-external _different_ we don't have to do anything drastic; just make _different_ anaphoric.

In [21]:
%%lamb
||book|| = L i_X9: Book_<e,t>
||Bob|| = L v_<e,<(X3,e),<Y2,t>>>: L i_X3: L o_(e,Y2): o[0] == Bob_e & v(o[0])((i,o[0]))(o[1])
# ||Fred|| = L v_<e,<(X4,e),<Y3,t>>>: L i_X4: L o_(e,Y3): v(Fred_e)((i,Fred_e))(o[1])
||Fred|| = L v_<e,<(X4,e),<Y3,t>>>: L i_X4: L o_(e,Y3): o[0] == Fred_e & v(o[0])((i,o[0]))(o[1])
||WarAndPeace|| = L v_<e,<(X5,e),<Y3,t>>>: L i_X5: L o_(e,Y3): o[0] == WaP_e & v(o[0])((i,o[0]))(o[1])
||read|| = L y_e: L x_e: L i_X7: L e_v: Read_<(e,e,v),t>(x,y,e)
# Here we go...
# ||different|| = L n_<X,<e,t>>: L i_X: L x_e: n(i)(x) & ~(x == Sel_<X,e>(i))
||different|| = L n_<((X,(Z,(e,X1))),Y),<e,t>>: L i_((X,(Z,(e,X1))),Y): L x_e: n(i)(x) & ~(x == i[0][1][1][0])

$[\![\text{\textbf{book}}]\!]^{}_{\left\langle{}X_{9},\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{X_{9}} \: . \: {Book}$<br />
$[\![\text{\textbf{Bob}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X''', e\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X''',\left\langle{}\left(e, Y''\right),t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X''', e\right),\left\langle{}Y'',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X'''} \: . \: \lambda{} o_{\left(e, Y''\right)} \: . \: ({Bob}_{e} = ({o}[\textsf{0}])) \wedge{} {v}({o}[\textsf{0}])({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$<br />
$[\![\text{\textbf{Fred}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{4}, e\right),\left\langle{}Y''',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X_{4},\left\langle{}\left(e, Y'''\right),t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{4}, e\right),\left\langle{}Y''',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X_{4}} \: . \: \lambda{} o_{\left(e, Y'''\right)} \: . \: ({Fred}_{e} = ({o}[\textsf{0}])) \wedge{} {v}({o}[\textsf{0}])({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$<br />
$[\![\text{\textbf{WarAndPeace}}]\!]^{}_{\left\langle{}\left\langle{}e,\left\langle{}\left(X_{5}, e\right),\left\langle{}Y''',t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X_{5},\left\langle{}\left(e, Y'''\right),t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X_{5}, e\right),\left\langle{}Y''',t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X_{5}} \: . \: \lambda{} o_{\left(e, Y'''\right)} \: . \: ({WaP}_{e} = ({o}[\textsf{0}])) \wedge{} {v}({o}[\textsf{0}])({i}, ({o}[\textsf{0}]))({o}[\textsf{1}])$<br />
$[\![\text{\textbf{read}}]\!]^{}_{\left\langle{}e,\left\langle{}e,\left\langle{}X_{7},\left\langle{}v,t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} y_{e} \: . \: \lambda{} x_{e} \: . \: \lambda{} i_{X_{7}} \: . \: \lambda{} e_{v} \: . \: {Read}({x}, {y}, {e})$<br />
$[\![\text{\textbf{different}}]\!]^{}_{\left\langle{}\left\langle{}\left(\left(X, \left(Z, \left(e, X'\right)\right)\right), Y\right),\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left(\left(X, \left(Z, \left(e, X'\right)\right)\right), Y\right),\left\langle{}e,t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}\left(\left(X, \left(Z, \left(e, X'\right)\right)\right), Y\right),\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(\left(X, \left(Z, \left(e, X'\right)\right)\right), Y\right)} \: . \: \lambda{} x_{e} \: . \: \neg{} ({x} = (((({i}[\textsf{0}])[\textsf{1}])[\textsf{1}])[\textsf{0}])) \wedge{} {n}({i})({x})$

In [22]:
counter += 1
exemplify("Bob read War and Peace. Fred read a different book.", counter, (C * (E * ((Bob * (b[2] * (WarAndPeace * (b[1] * (t[2] * (read * t[1])))))) * (aND * (Fred * (b[4] * ((a * (different * book)) * (b[3] * (t[4] * (read * t[3])))))))))))

(14)	Bob read War and Peace. Fred read a different book.

[C [E [[Bob [2 [WarAndPeace [1 [t2 [read t1]]]]]] [aND [Fred [4 [[a [different book]] [3 [t4 [read t3]]]]]]]]]]



(Exists m_((e,(e,v)),(e,(e,v))): ((((((~((((m_((e,(e,v)),(e,(e,v)))[0])[1])[0]) <=> (((m_((e,(e,v)),(e,(e,v)))[1])[1])[0])) & (Bob_e <=> ((m_((e,(e,v)),(e,(e,v)))[0])[0]))) & (Fred_e <=> ((m_((e,(e,v)),(e,(e,v)))[1])[0]))) & (WaP_e <=> (((m_((e,(e,v)),(e,(e,v)))[0])[1])[0]))) & Book_<e,t>(((m_((e,(e,v)),(e,(e,v)))[1])[1])[0])) & Read_<(e,e,v),t>(((m_((e,(e,v)),(e,(e,v)))[0])[0]), (((m_((e,(e,v)),(e,(e,v)))[0])[1])[0]), (((m_((e,(e,v)),(e,(e,v)))[0])[1])[1]))) & Read_<(e,e,v),t>(((m_((e,(e,v)),(e,(e,v)))[1])[0]), (((m_((e,(e,v)),(e,(e,v)))[1])[1])[0]), (((m_((e,(e,v)),(e,(e,v)))[1])[1])[1]))))

#### Adding a presupposition

Let's add a pressupposition to the effect that the thing anphorically referred to by _different_ is also a book. We do these by means of the [partiality](https://github.com/rawlins/lambda-notebook/blob/master/notebooks/documentation/Partiality%20documentation.ipynb) operator.

In [23]:
%%lamb
||different|| = L n_<((X,(Z,(e,X1))),Y),<e,t>>: L i_((X,(Z,(e,X1))),Y): L x_e: n(i)(x) & Partial(~(x == i[0][1][1][0]), n(i)(i[0][1][1][0]))

$[\![\text{\textbf{different}}]\!]^{}_{\left\langle{}\left\langle{}\left(\left(X, \left(Z, \left(e, X'\right)\right)\right), Y\right),\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left(\left(X, \left(Z, \left(e, X'\right)\right)\right), Y\right),\left\langle{}e,t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}\left(\left(X, \left(Z, \left(e, X'\right)\right)\right), Y\right),\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(\left(X, \left(Z, \left(e, X'\right)\right)\right), Y\right)} \: . \: \lambda{} x_{e} \: . \: {n}({i})({x}) \wedge{} \left|\begin{array}{l}\neg{} ({x} = (((({i}[\textsf{0}])[\textsf{1}])[\textsf{1}])[\textsf{0}]))\\{n}({i})(((({i}[\textsf{0}])[\textsf{1}])[\textsf{1}])[\textsf{0}])\end{array}\right|$

In [24]:
print(f"({counter})\t...")
(C * (E * ((Bob * (b[2] * (WarAndPeace * (b[1] * (t[2] * (read * t[1])))))) * (aND * (Fred * (b[4] * ((a * (different * book)) * (b[3] * (t[4] * (read * t[3])))))))))).content[0].content

(14)	...


(Exists m_((e,(e,v)),(e,(e,v))): (((((((Bob_e <=> ((m_((e,(e,v)),(e,(e,v)))[0])[0])) & (Fred_e <=> ((m_((e,(e,v)),(e,(e,v)))[1])[0]))) & (WaP_e <=> (((m_((e,(e,v)),(e,(e,v)))[0])[1])[0]))) & Book_<e,t>(((m_((e,(e,v)),(e,(e,v)))[1])[1])[0])) & Read_<(e,e,v),t>(((m_((e,(e,v)),(e,(e,v)))[0])[0]), (((m_((e,(e,v)),(e,(e,v)))[0])[1])[0]), (((m_((e,(e,v)),(e,(e,v)))[0])[1])[1]))) & Read_<(e,e,v),t>(((m_((e,(e,v)),(e,(e,v)))[1])[0]), (((m_((e,(e,v)),(e,(e,v)))[1])[1])[0]), (((m_((e,(e,v)),(e,(e,v)))[1])[1])[1]))) & Partial(~((((m_((e,(e,v)),(e,(e,v)))[0])[1])[0]) <=> (((m_((e,(e,v)),(e,(e,v)))[1])[1])[0])), Book_<e,t>(((m_((e,(e,v)),(e,(e,v)))[0])[1])[0]))))

Asserted:

- Bob read _War and Peace_.
- Fred read a book that isn't _War and Peace_.

Presupposed:

- _War and Peace_ is a book.

This seems right.

### Sentence-internal

For sentence-internal _different_ we have to mess around with the definition of the determiner, to ensure that the function that it defines is passed to the anaphoric context for its second argument; $[\![\textbf{different}]\!]$ is then anaphoric on that function. There is some fiddling here: _every student read a book_ defines a function $f$ mapping students to a pair consisting of a book and an event of that student reading that book; to be precise, in _every student read a different book_ we want it to be $\lambda x.f(x)[0]$ that is injective, i.e. the function from students to books that they read. We also incorporate a presupposition via partiality into the lexical entry, constraining the anaphoric resolution as follows:

$$
[\![\textbf{different}]\!] = \lambda n_{\langle(X,e,\langle e,(e,Y)\rangle),\langle e,t\rangle\rangle}.\lambda i_{(X,e,\langle e,(e,Y)\rangle)}.\lambda x_e.n(i)(x)\wedge\left|\substack{injective\big(Sel_{\langle(X,e,\langle e,(e,Y)\rangle),\langle e,e\rangle\rangle}(i)\big)\\Sel_{\langle(X,e,\langle e,(e,Y)\rangle),\langle e,e\rangle\rangle}(i)\big(Sel_{\langle(X,e,\langle e,(e,Y)\rangle),e\rangle}(i)\big) = x}\right|
$$

The lexical entry for _different_ given below is such that the anaphoric resolutions are correct, and so the presupposition is satisfied, i.e. where

$$
\begin{align*}
Sel_{\langle(X,e,\langle e,(e,Y)\rangle),\langle e,e\rangle\rangle}&:=\lambda u_{(X,e,\langle e,(e,Y)\rangle)}.\lambda x_e.u[2](x)[0],\mbox{ and}\\
Sel_{\langle(X,e,\langle e,(e,Y)\rangle),e\rangle}&:=\lambda u_{(X,e,\langle e,(e,Y)\rangle)}.u[1]
\end{align*}
$$

N.B., for simplicity's sake, in this section I've adapted the (DRT-esque) lexical entry for _every_ from the 2017 paper. The more complicated one from the 2019 paper would work just as well, but the extra complexicty might obscure the presentation.

In [25]:
%%lamb
||student|| = L i_X: Student_<e,t>
||every|| = L n_<X,<e,t>>: L v_<e,<(X,e,<e,Y>),<Y,t>>>: L i_X: L f_<e,Y>: Forall x_e: n(i)(x) ==> v(x)((i,x,f))(f(x))
injective = L f_<X9,Y9>: Forall a_X9: Forall b_X9: Forall c_Y9: ((f(a) == c) & (f(b) == c)) ==> (a == b)
# ||different|| = L n_<X1,<e,t>>: L i_X1: L x_e: n(i)(x) & injective(Sel_<X1,<e,e>>(i))
||different|| = L n_<(X1,e,<e,(e,Y9)>),<e,t>>: L i_(X1,e,<e,(e,Y9)>): L x_e: n(i)(x) & Partial(injective(L x_e: i[2](x)[0]), i[2](i[1])[0] == x)

$[\![\text{\textbf{student}}]\!]^{}_{\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{}} \:=\: \lambda{} i_{X} \: . \: {Student}$<br />
$[\![\text{\textbf{every}}]\!]^{}_{\left\langle{}\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left\langle{}e,\left\langle{}\left(X, e, \left\langle{}e,Y\right\rangle{}\right),\left\langle{}Y,t\right\rangle{}\right\rangle{}\right\rangle{},\left\langle{}X,\left\langle{}\left\langle{}e,Y\right\rangle{},t\right\rangle{}\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}X,\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} v_{\left\langle{}e,\left\langle{}\left(X, e, \left\langle{}e,Y\right\rangle{}\right),\left\langle{}Y,t\right\rangle{}\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{X} \: . \: \lambda{} f_{\left\langle{}e,Y\right\rangle{}} \: . \: \forall{} x_{e} \: . \: {n}({i})({x}) \rightarrow{} {v}({x})({i}, {x}, {f})({f}({x}))$<br />
${injective}_{\left\langle{}\left\langle{}X_{9},Y_{9}\right\rangle{},t\right\rangle{}}\:=\:\lambda{} f_{\left\langle{}X_{9},Y_{9}\right\rangle{}} \: . \: \forall{} a_{X_{9}} \: . \: \forall{} b_{X_{9}} \: . \: \forall{} c_{Y_{9}} \: . \: (({c} = {f}({a})) \wedge{} ({c} = {f}({b}))) \rightarrow{} {a} = {b}$<br />
$[\![\text{\textbf{different}}]\!]^{}_{\left\langle{}\left\langle{}\left(X', e, \left\langle{}e,\left(e, Y_{9}\right)\right\rangle{}\right),\left\langle{}e,t\right\rangle{}\right\rangle{},\left\langle{}\left(X', e, \left\langle{}e,\left(e, Y_{9}\right)\right\rangle{}\right),\left\langle{}e,t\right\rangle{}\right\rangle{}\right\rangle{}} \:=\: \lambda{} n_{\left\langle{}\left(X', e, \left\langle{}e,\left(e, Y_{9}\right)\right\rangle{}\right),\left\langle{}e,t\right\rangle{}\right\rangle{}} \: . \: \lambda{} i_{\left(X', e, \left\langle{}e,\left(e, Y_{9}\right)\right\rangle{}\right)} \: . \: \lambda{} x_{e} \: . \: {n}({i})({x}) \wedge{} \left|\begin{array}{l}{[\lambda{} f_{\left\langle{}e,e\right\rangle{}} \: . \: \forall{} a_{e} \: . \: \forall{} b_{e} \: . \: \forall{} c_{e} \: . \: (({c} = {f}({a})) \wedge{} ({c} = {f}({b}))) \rightarrow{} {a} = {b}]}(\lambda{} x_{e} \: . \: ({i}[\textsf{2}])({x})[\textsf{0}])\\{x} = (({i}[\textsf{2}])({i}[\textsf{1}])[\textsf{0}])\end{array}\right|$

In [26]:
counter += 1
exemplify("Every student read a different book.", counter, (C * (E * ((every * student) * (b[4] * ((a * (different * book)) * (b[3] * (t[4] * (read * t[3])))))))))

(15)	Every student read a different book.

[C [E [[every student] [4 [[a [different book]] [3 [t4 [read t3]]]]]]]]



(Exists m_<e,(e,v)>: (Forall x_e: (Student_<e,t>(x_e) >> ((Book_<e,t>(m_<e,(e,v)>(x_e)[0]) & Read_<(e,e,v),t>(x_e, (m_<e,(e,v)>(x_e)[0]), (m_<e,(e,v)>(x_e)[1]))) & (Forall a_e: (Forall b_e: (Forall c_e: (((c_e <=> (m_<e,(e,v)>(a_e)[0])) & (c_e <=> (m_<e,(e,v)>(b_e)[0]))) >> (a_e <=> b_e)))))))))

This time, no presupposition projects. The compositional system has grapsed that the presupposition is satisfied, since it reduces to $m(x)[0]=m(x)[0]$.